# 04 — Gabor vs Gaussian-derivative comparison

Fits Gabor models to the 31 verified m=1 simple cells under matched optimisation constraints and compares them to the Gaussian-derivative fits. Saves results to `derived_data/m1_cells/gabor_vs_gd_m1_31_v3.csv`.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

gallery_dir  = REPO_ROOT / 'derived_data' / 'm1_cells'
DATASET_PATH = gallery_dir / 'm1_neuron_dataset.pkl'
cache_dir    = REPO_ROOT / 'data' / 'cache'
OUT_DIR      = gallery_dir

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rf_analysis.sparse_noise import fit_rf_by_order
from rf_analysis.gabor_model  import fit_gabor, K_GABOR

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

N_RANDOM_STARTS = 15
SMOOTH_SIGMA    = 0.75
RANDOM_SEED     = 20260831

with open(DATASET_PATH, 'rb') as f:
    dataset = pickle.load(f)



In [ ]:
def subfields(rec):
    rf = np.asarray(rec['rf_raw'], dtype=float)
    return np.clip(rf, 0, None), np.clip(-rf, 0, None)

_on, _off = subfields(dataset[0])

In [ ]:
rows = []

for i, rec in enumerate(dataset, start=1):
    seed = RANDOM_SEED + int(rec['cell_id']) % 100000
    rf_on, rf_off = subfields(rec)
    px = float(rec['pixel_size_deg'])

    p_gd, q_gd = fit_rf_by_order(rf_on, rf_off, m=1,
                                 pixel_size_deg=px, smooth_sigma=SMOOTH_SIGMA,
                                 n_random_starts=N_RANDOM_STARTS, seed=seed)
    p_gb, q_gb = fit_gabor(rf_on, rf_off,
                           pixel_size_deg=px, smooth_sigma=SMOOTH_SIGMA,
                           n_random_starts=N_RANDOM_STARTS, seed=seed)

    rows.append({
        'name': rec['name'], 'cell_id': int(rec['cell_id']),
        'container_id': int(rec['container_id']),
        'r2_pipeline': float(rec.get('r_squared', np.nan)),
        'r2_gd':    q_gd['r_squared'],  'r2_gabor':   q_gb['r_squared'],
        'rmse_gd':  q_gd['rmse'],       'rmse_gabor': q_gb['rmse'],
        'aic_gd':   q_gd['aic'],        'aic_gabor':  q_gb['aic'],
        'delta_aic': q_gb['aic'] - q_gd['aic'],
        'gabor_better_r2':  q_gb['r_squared'] > q_gd['r_squared'],
        'gabor_better_aic': q_gb['aic'] < q_gd['aic'],
        'kappa_dir_gd':    p_gd['kappa_dir'],  'kappa_dir_gabor': p_gb['kappa_dir'],
        'sigma_gd':        p_gd['sigma'],      'sigma_gabor':     p_gb['sigma'],
        'gabor_period_deg': p_gb['carrier_period_deg'],
        'r2_improve_gd':    p_gd['r2_improvement'],
        'r2_improve_gabor': p_gb['r2_improvement'],
    })

res = pd.DataFrame(rows)


In [ ]:
TOL = 0.01
res['gd_deficit'] = res['r2_pipeline'] - res['r2_gd']
bad = res[res['gd_deficit'] > TOL]

if len(bad):
    pass
else:
    pass



In [ ]:
n = len(res)
med_gd, med_gab = res['r2_gd'].median(), res['r2_gabor'].median()
n_gab_r2  = int(res['gabor_better_r2'].sum())
n_gab_aic = int(res['gabor_better_aic'].sum())
med_daic  = res['delta_aic'].median()


sign = '+' if med_daic >= 0 else ''


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5.2))
lims = [max(0.0, float(min(res['r2_gd'].min(), res['r2_gabor'].min())) - 0.05),
        min(1.0, float(max(res['r2_gd'].max(), res['r2_gabor'].max())) + 0.05)]

a = ax[0]
a.scatter(res['r2_gd'], res['r2_gabor'], s=55, c='#1a7a4a',
          edgecolor='white', linewidth=0.6, zorder=3)
a.plot(lims, lims, '--', color='#333', lw=1)
a.set(xlim=lims, ylim=lims,
      xlabel='$R^2$  first-order Gaussian derivative  ($k=7$)',
      ylabel=f'$R^2$  Gabor  ($k={K_GABOR}$)')
a.set_title(f'Explained variance\nGabor ahead in {n_gab_r2}/{n}', fontweight='bold')

a = ax[1]
a.scatter(res['kappa_dir_gd'], res['kappa_dir_gabor'], s=55, c='#c0392b',
          edgecolor='white', linewidth=0.6, zorder=3)
klim = [0.3, float(max(res['kappa_dir_gd'].max(),
                       res['kappa_dir_gabor'].max())) * 1.05]
a.plot(klim, klim, '--', color='#333', lw=1)
a.axhline(1, color='#888', lw=0.7); a.axvline(1, color='#888', lw=0.7)
a.set(xlim=klim, ylim=klim,
      xlabel=r'$\kappa_{\mathrm{dir}}$  Gaussian derivative',
      ylabel=r'$\kappa_{\mathrm{dir}}$  Gabor')
a.set_title('Shape estimate is model-dependent', fontweight='bold')

fig.suptitle('Goodness of fit does not separate the two families',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig07_gaussian_vs_gabor.png', dpi=300, bbox_inches='tight')
plt.savefig(OUT_DIR / 'fig07_gaussian_vs_gabor.pdf', bbox_inches='tight')
plt.show()

In [ ]:
csv_path = OUT_DIR / 'gabor_vs_gd_m1_31_v3.csv'
res.round(4).to_csv(csv_path, index=False)
